# View LSSTCam Single Visit in Firefly

- author Sylvie Dagoret-Campagne
- creation date 2026-04-04

- quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/current/drp_pipe/
- DRP runs: https://rubinobs.atlassian.net/browse/DM-53881

```
LSSTCam DP2 DRP using v30_0_0
Détails clés
Description
    repo = dp2_prep
    collection
        stage1: LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1

            individual epochs (only step1a+step1b results are included here):

                LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1/epoch1

                LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1/epoch2

                LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1/epoch3

        stage2: LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2

        stage3: LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3

        stage4: LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4

        stage3 rerun: LSSTCam/runs/DRP/DP2/v30_0_6_rc1/DM-53881/stage3

    skymap ='lsst_cells_v2'

    input collections: 

        epoch1: LSSTCam/raw/DP2/20250424-20250702/DM-53733

        epoch2: LSSTCam/raw/DP2/20250703-20250921/DM-53733

        epoch3: LSSTCam/raw/DP2/20251024-20260106/DM-53733

    dataQuery: "instrument='LSSTCam' AND skymap='lsst_cells_v2' (2229 tract list:   includes Abell)

    pipeline definition yaml file ${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml

    submission directory:  /sdf/data/rubin/shared/campaigns/LSSTCam/DM-53881

    Plot Navigator 

        stage1

        stage2

        stage3

        stage4

    Metrics have identifier=LSSTCam/DRP and timestamp=20260123000000Z

        For stage3 rerun timestamp=20260404T000000Z
```

## Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler

import lsst.geom as geom
from lsst.geom import SpherePoint, degrees
import lsst.afw.display as afwDisplay
from firefly_client import FireflyClient
import firefly_client.plot as ffplt

from lsst.skymap import PatchInfo, Index2D

In [ ]:
import gc

In [ ]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

In [ ]:
def get_first_existing_dataset(butler, dataset_names, dataId, collections=None):
    for name in dataset_names:
        try:
            obj = butler.get(name, dataId, collections=collections)
            print(f"✔ Found dataset: {name}")
            return obj, name
        except Exception:
            continue

    raise ValueError("No valid dataset found among candidates.")

## Config

In [ ]:
# The output repo is tagged with the Jira ticket number "DM-53881":
repo = "dp2_prep"

COLLECTIONS = [
    "LSSTCam/defaults",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]


COLLECTIONS = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    # "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1/epoch1",
    # "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1/epoch2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1/epoch3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
]
# "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
# "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
# "LSSTCam/runs/DRP/DP2/v30_0_6_rc1/DM-53881/stage3",


INSTRUMENT = "LSSTCam"
skymapName = "lsst_cells_v2"
where_clause = "instrument = '" + INSTRUMENT + "'"
collectionStr = COLLECTIONS[0].replace("/", "_")
BANDSEL = "r"  # Most fields were observed in red filter

In [ ]:
# Initialize the butler repo:
butler = Butler(repo, collections=COLLECTIONS)
registry = butler.registry

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=COLLECTIONS)

In [ ]:
camera = butler.get("camera", collections=COLLECTIONS, instrument=INSTRUMENT)

In [ ]:
# print(camera.getName(),camera.getNameMap())

In [ ]:
FLAG_DUMP_DATASETS = True
if FLAG_DUMP_DATASETS:
    for datasetType in registry.queryDatasetTypes():
        if registry.queryDatasets(datasetType, collections=COLLECTIONS).any(execute=False, exact=False):
            # Skip pipeline bookkeeping products
            if (
                ("_config" not in datasetType.name)
                and ("_log" not in datasetType.name)
                and ("_metadata" not in datasetType.name)
                and ("_resource_usage" not in datasetType.name)
                and ("Plot" not in datasetType.name)
                and ("Metric" not in datasetType.name)
                and ("metric" not in datasetType.name)
            ):
                print(datasetType)

In [ ]:
for dt in registry.queryDatasetTypes():
    dims = dt.dimensions.names
    if (
        "visit" in dims
        and "detector" in dims
        and "log" not in dt.name
        and "metadata" not in dt.name
        and "metrics" not in dt.name
    ):
        print(dt.name, dims)

In [ ]:
dt = registry.getDatasetType("visit_image")
print(dt)
print(dt.dimensions)

In [ ]:
dt = registry.getDatasetType("calexp")
print(dt)
print(dt.dimensions)

## Find the visits in the collections

In [ ]:
visitsTable = butler.get("preliminary_visit_table", collections=COLLECTIONS)

In [ ]:
visitsTable

In [ ]:
visits_table = set(visitsTable["visit"])

## Load the visits to fetch

In [ ]:
FULL_FILENAME_FROM_BROKER = "data_fromFink/alerts_dect94.csv"

In [ ]:
df = pd.read_csv(FULL_FILENAME_FROM_BROKER)

In [ ]:
df

## Check which of found Fink visits are in the visitTable

In [ ]:
df["visit_in_table"] = df["visit"].isin(visits_table)

In [ ]:
print(df["visit_in_table"].value_counts())

In [ ]:
missing_visits = df.loc[~df["visit_in_table"], "visit"].unique()
print("Visites absentes :", missing_visits)

In [ ]:
selected_visits = df.loc[df["visit_in_table"], "visit"].unique()
print("Visites presentes :", selected_visits)

In [ ]:
visit_df = visitsTable.to_pandas()

# garder uniquement info utile
visit_df = visit_df[["visit", "band"]]

# supprimer doublons (important)
visit_df = visit_df.drop_duplicates(subset=["visit"])

In [ ]:
df_filtered = df.merge(visit_df, on="visit", how="inner", suffixes=("", "_visit"))

In [ ]:
df_filtered

In [ ]:
df_filtered = df_filtered.drop_duplicates(subset=["visit", "detector", "band"])

In [ ]:
df_filtered

### Search for Single Visit Image in Butler

In [ ]:
DATASET_CANDIDATES = [
    "visit_image",
    "calexp",
    "preliminary_visit_image",
    "difference_image_predetection",
    "postISRCCD",
]

images = []

for idx, row in df_filtered.iterrows():
    dataId = {
        "instrument": INSTRUMENT,
        "visit": int(row["visit"]),
        "detector": int(row["detector"]),
        "band": row["band"],
    }

    print(f"\n[TRY] idx={idx} → {dataId}")

    img = None

    for datasetType in DATASET_CANDIDATES:
        try:
            img = butler.get(datasetType, dataId, collections=COLLECTIONS)
            print(f"[OK] idx={idx} → {datasetType}")
            break

        except Exception as e:
            print(f"[FAIL] {datasetType} → {type(e).__name__}")

    if img is None:
        print(f"[ERROR] idx={idx} → aucun dataset trouvé")
    else:
        images.append(img)

In [ ]:
def dataset_exists(datasetType, dataId):
    return registry.queryDatasets(datasetType, dataId=dataId, collections=COLLECTIONS).any(execute=True)

In [ ]:
for datasetType in DATASET_CANDIDATES:
    try:
        if not dataset_exists(datasetType, dataId):
            continue

        img = butler.get(datasetType, dataId, collections=COLLECTIONS)
        print(f"[OK] idx={idx} → {datasetType}")
        break

    except Exception:
        continue

In [ ]:
datasetType = "preliminary_visit_image"

## Load  image and plot

### Extract the image from butler

In [ ]:
bands = ["u", "g", "r", "i", "z", "y"]

In [ ]:
all_singlevisitimages = []
all_titles = []
all_datapoints = []
for idx, row in df_filtered.iterrows():
    the_dataId = {
        "instrument": INSTRUMENT,
        "visit": int(row["visit"]),
        "detector": int(row["detector"]),
        "band": row["band"],
    }
    x, y = row["x"], row["y"]
    list_of_points = [(x, y)]
    try:
        the_singlevisitimage = butler.get(datasetType, the_dataId)

    except Exception as e:
        # log utile mais compact
        print(f"[FAIL] idx={idx} → visit={row['visit']} det={row['detector']} band={row['band']}")
        print(f"       {type(e).__name__}: {e}")
        continue  # on passe à la ligne suivante

    # seulement si succès
    the_title = f"{int(row['visit'])}:{row['band']}"

    all_singlevisitimages.append(the_singlevisitimage)
    all_titles.append(the_title)
    all_datapoints.append(list_of_points)

### Plot

In [ ]:
afwDisplay.setDefaultBackend("firefly")
# display = afwDisplay.Display(frame=1)
# display.scale("asinh", "zscale")
# display.mtv(image, title = target_title)

In [ ]:
N = len(all_singlevisitimages)
print(f"Number of image to plot : {N}")
for count in range(N):
    display = afwDisplay.Display(frame=count + 1)
    # cannot succeed to show white stars on dark sky
    # display.setImageColormap('gray')
    display.scale("asinh", "zscale")
    display.mtv(all_singlevisitimages[count].image, title=all_titles[count])
    list_of_points = all_datapoints[count]
    with display.Buffering():
        for x, y in list_of_points:
            display.dot("o", x, y, size=20, ctype="red")

In [ ]:
# display.clearViewer()